In [1]:
import numpy as np
import pandas as pd
import sklearn
import sklearn.preprocessing
import sklearn.feature_selection
pd.options.mode.chained_assignment ='warn' # default='warn'
import graphviz

#Leemos los datos
datos=pd.read_csv('futbol_uruguayo.csv')
datos.head(2)

,home,away,date,gh,ga,full_time,competition,home_ident,away_ident,home_country,away_country,home_code,away_code,home_continent,away_continent,continent,level
0,Bella Vista,Defensor Sporting,1932-03-05,1.0,2.0,F,uruguay,Bella Vista (Uruguay),Defensor Sporting (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
1,CA Penarol,River Plate,1932-03-05,1.0,1.0,F,uruguay,CA Penarol (Uruguay),River Plate (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national


In [2]:
# PRE PROCESAMIENTOO
# CREAMOS LA VARIABLE DE SALIDA "RESULTADO"

# Definimos las condiciones
condiciones = [
    datos["gh"] > datos["ga"],  # Gana el local
    datos["gh"] < datos["ga"],  # Gana el visitante
]

# Definimos los resultados para cada condición
opciones = ["gana el local", "gana el visitante"]

# Aplicamos las condiciones; si no se cumple ninguna (son iguales), asigna "hay empate"
datos["resultado"] = np.select(condiciones, opciones, default="hay empate")

# Ejemplo de visualización
datos[["gh", "ga", "resultado"]].head(2)

,gh,ga,resultado
0,1.0,2.0,gana el visitante
1,1.0,1.0,hay empate


In [3]:
# ELIMINAMOS FILAS DUPLICADAS
cant_duplicados = datos.duplicated().sum()
print(f"Filas duplicadas encontradas: {cant_duplicados}")
datos = datos.drop_duplicates()

# DETECTAMOS DATOS FALTANTES E IMPUTAMOS MEDIA

print("\nValores faltantes por columna:")
print(datos.isnull().sum())

# Seleccionar columnas numEricas e imputar la media si existen valores nulos
columnas_numericas = datos.select_dtypes(include=["number"]).columns

for col in columnas_numericas:
    if datos[col].isnull().sum() > 0:
        media_col = datos[col].mean()
        datos[col] = datos[col].fillna(media_col)
        print(f"Se imputó la media ({media_col:.2f}) en la columna '{col}'")

Filas duplicadas encontradas: 1

Valores faltantes por columna:
home              0
away              0
date              0
gh                0
ga                0
full_time         0
competition       0
home_ident        0
away_ident        0
home_country      0
away_country      0
home_code         0
away_code         0
home_continent    0
away_continent    0
continent         0
level             0
resultado         0
dtype: int64


In [4]:
# Creamos nuevos atributos para usar en el algoritmo a partir del atributo fecha

# 1. Convertir la columna a tipo datetime
datos["date"] = pd.to_datetime(datos["date"])

# 2. Extraer componentes temporales relevantes

# Año (para capturar tendencias históricas, ej: evolución del fútbol con las décadas)
datos["year"] = datos["date"].dt.year

# Mes y Trimestre (para capturar estacionalidad en el torneo)
datos["month"] = datos["date"].dt.month

# Día de la semana (0=Lunes, 6=Domingo; útil para ver si se juega fin de semana o entre semana)
datos["dayofweek"] = datos["date"].dt.dayofweek

# Indicador de si es fin de semana (1 si es Sábado/Domingo, 0 si no)
datos["is_weekend"] = datos["dayofweek"].isin([5, 6]).astype(int)

datos.head(2)

,home,away,date,gh,ga,full_time,competition,home_ident,away_ident,home_country,...,away_code,home_continent,away_continent,continent,level,resultado,year,month,dayofweek,is_weekend
0,Bella Vista,Defensor Sporting,1932-03-05,1.0,2.0,F,uruguay,Bella Vista (Uruguay),Defensor Sporting (Uruguay),uruguay,...,UY,South America,South America,South America,national,gana el visitante,1932,3,5,1
1,CA Penarol,River Plate,1932-03-05,1.0,1.0,F,uruguay,CA Penarol (Uruguay),River Plate (Uruguay),uruguay,...,UY,South America,South America,South America,national,hay empate,1932,3,5,1


In [5]:
# se particiona los datos en X y Y
datos_y = datos[["resultado"]]
datos_x = datos.drop(["resultado"], axis=1, inplace=False)
datos_x = datos[["home", "away", "full_time", "year", "month", "dayofweek", "is_weekend"]]

#DIVISION DEL CONJUNTO DE DATOS ENTRENAMIENTO Y PRUEBA
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    datos_x, datos_y, test_size=0.2, random_state=42)

display(X_train.head(5))
display(y_train.head(5))
display(X_train.shape)
display(y_train.shape)

,home,away,full_time,year,month,dayofweek,is_weekend
12093,El Tanque Sisley,Defensor Sporting,F,2014,4,6,1
13404,Montevideo Wanderers,Plaza Colonia,F,2019,5,6,1
6778,Bella Vista,Montevideo Wanderers,F,1990,6,5,1
8478,River Plate,Danubio,F,2000,6,1,0
10251,Club Sportivo Cerrito,Rentistas,F,2006,11,6,1


,resultado
12093,hay empate
13404,hay empate
6778,gana el local
8478,hay empate
10251,hay empate


(12164, 7)

(12164, 1)

In [65]:
display(datos_x.head(5))

,home,away,full_time,year,month,dayofweek,is_weekend
0,Bella Vista,Defensor Sporting,F,1932,3,5,1
1,CA Penarol,River Plate,F,1932,3,5,1
2,Montevideo Wanderers,Racing Club,F,1932,3,5,1
3,Central Espanol,Rampla Juniors Futbol Club,F,1932,3,5,1
4,Nacional,Institucion Atletica Sud America,F,1932,3,5,1


In [6]:
#TRATAMIENTO DE ATRIBUTOS CATEGORICOS

#opcion 1: codificar con valores discretos. Problema: presupone un orden
#opcion 2: one-hot encoding. Problema: aumenta la dimensionalidad de los dato. Optamos por esta.


from sklearn.preprocessing import OneHotEncoder

# Aplicamos One-Hot Encoder aprendiendo solo del conjunto de entrenamiento
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

# Ajustar con train y transformar train
X_train_encoded = encoder.fit_transform(X_train[["home", "away", "full_time", "year", "month", "dayofweek", "is_weekend"]])

# Transformar test (sin hacer fit)
X_test_encoded = encoder.transform(X_test[["home", "away", "full_time", "year", "month", "dayofweek", "is_weekend"]])

# ahora convertimos a Y en categorica para ahorrar memoria, aunque no es estrictamente neesario
mapa_resultados = {
    "gana el local": 0,
    "gana el visitante": 1,
    "hay empate": 2,
}

# Seleccionamos la columna 'resultado'
y_train_encoded = y_train["resultado"].map(mapa_resultados)
y_test_encoded = y_test["resultado"].map(mapa_resultados)

In [82]:
y_test_encoded.head(5)

566      0
6712     2
14914    0
6369     1
9380     0
Name: resultado, dtype: int64

In [83]:
from sklearn.tree import DecisionTreeClassifier

modelo = DecisionTreeClassifier(random_state=42)
modelo.fit(X_train_encoded, y_train_encoded)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [ ]:
# ==============================================================================
# IMPLEMENTACIÓN DEL ÁRBOL DE DECISIÓN CON GANANCIA MÍNIMA DE INFORMACIÓN
# ==============================================================================

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

# Defina el valor de la ganancia mínima de información deseada 
min_info_gain = 0.01

# Inicializar el Árbol de Decisión:
# - criterion='entropy': calcula la Ganancia de Información (Information Gain)
# - min_impurity_decrease=min_info_gain: detiene la división cuando la ganancia es menor al valor especificado
arbol_modelo = DecisionTreeClassifier(
    criterion="entropy", min_impurity_decrease=min_info_gain, random_state=42
)

# Entrenar el modelo
arbol_modelo.fit(X_train_encoded, y_train_encoded)


,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'entropy'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.01
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively in

In [8]:

# ==============================================================================
# EVALUACIÓN DEL MODELO SOBRE EL CONJUNTO DE PRUEBA
# ==============================================================================
y_pred = arbol_modelo.predict(X_test_encoded)

print(
    f"=== Árbol de Decisión (min_info_gain = {min_info_gain}) ==="
)
print(f"Exactitud (Accuracy): {accuracy_score(y_test_encoded, y_pred):.4f}\n")

print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred))

# Matriz de Confusión
cm = confusion_matrix(y_test_encoded, y_pred, labels=["L", "V", "E"])

plt.figure(figsize=(6, 4.5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=["L", "V", "E"],
    yticklabels=["L", "V", "E"],
)
plt.title("Matriz de Confusión - Árbol de Decisión")
plt.xlabel("Predicción ('L', 'V', 'E')")
plt.ylabel("Valor Real ('L', 'V', 'E')")
plt.show()

=== Árbol de Decisión (min_info_gain = 0.01) ===
Exactitud (Accuracy): 0.4875

Reporte de Clasificación:


ValueError: Mix of label input types (string and number); Got ['gana el local' 'gana el visitante' 'hay empate'] and [0 1].